# Урок 19. Компьютерные сети и протоколы

11 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 18](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-18.ipynb) · [Урок 20 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-20.ipynb)

---

Модель уровней, стек TCP/IP. Коммутация пакетов. Основные протоколы. Расчёт времени передачи и пропускной способности.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
for адрес in ("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py",
              "https://cdn.jsdelivr.net/gh/LeksssGray/informatika-7-11@main/lib/schoolinf.py"):
    try:
        urllib.request.urlretrieve(адрес, "schoolinf.py")
        break
    except Exception as ошибка:
        print("⚠️ Не скачалась библиотека проверки:", ошибка)
else:
    raise SystemExit("Проверь интернет и запусти эту ячейку ещё раз. "
                     "Не помогло — покажи это сообщение учителю.")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-19", name=ФИО, klass=Класс)

## Разбираемся

### Сеть как слоёный пирог

Когда вы открываете страницу, работают десятки устройств и программ,
написанных разными людьми в разные десятилетия. Договориться обо всём
сразу невозможно — поэтому сеть построена **уровнями**: каждый решает
свою задачу и не лезет в чужие.

| Уровень | Чем занимается | Примеры |
|---|---|---|
| прикладной | смысл данных | HTTP, SMTP, DNS, FTP |
| транспортный | доставка потока между программами | TCP, UDP |
| сетевой | доставка пакета между узлами | IP |
| канальный | передача по конкретной линии | Ethernet, Wi-Fi |

Главная идея: **каждый уровень пользуется нижним и не знает, как тот
устроен**. HTTP всё равно, идут данные по оптике или по радио,
а Wi-Fi всё равно, что внутри пакета — письмо или видео.

Благодаря этому Wi-Fi появился в 1997 году, не сломав протоколы
1970-х.

### TCP и UDP

На транспортном уровне два протокола с разной философией.

| | TCP | UDP |
|---|---|---|
| Соединение | устанавливается | нет |
| Гарантия доставки | есть, переспрашивает | нет |
| Порядок пакетов | восстанавливается | как придут |
| Накладные расходы | выше | минимальные |
| Где применяют | веб, почта, файлы | видеозвонки, игры, DNS |

Почему для звонка выбирают ненадёжный UDP? Потому что переспрашивать
потерянный кусок звука бессмысленно: пока он придёт, разговор уйдёт
вперёд. Лучше щелчок, чем задержка.

### Коммутация пакетов

Данные режутся на пакеты; каждый едет самостоятельно, возможно,
своим маршрутом; получатель собирает их по номерам.

Альтернатива — **коммутация каналов**, как в старой телефонии:
на время разговора выделяется линия целиком. Надёжно, предсказуемо
и расточительно: в паузах линия простаивает.

Интернет выбрал пакеты, потому что они позволяют делить один канал
между всеми сразу.

### Адреса и имена

**IP-адрес** — числовой адрес узла: `192.168.1.10`.
**Доменное имя** — человеческое: `example.com`.
**DNS** — служба, которая переводит имя в адрес.

Открывая сайт, браузер сначала спрашивает у DNS адрес, и только
потом обращается к серверу. Поэтому «интернет не работает, но
по IP всё открывается» — типичная поломка DNS.

### Порты

IP-адрес приводит пакет к компьютеру, а **порт** — к нужной
программе на нём.

| Порт | Служба |
|---|---|
| 80 | HTTP |
| 443 | HTTPS |
| 25 | SMTP (почта) |
| 53 | DNS |
| 22 | SSH |

Пара «адрес : порт» и есть точный адресат в сети.

### Расчёт времени передачи

Формула та же, что в девятом классе:

$$t = \frac{I}{v}$$

Но в задачах ЕГЭ добавляются тонкости:

* **скорость в битах, объём в байтах** — множитель 8;
* иногда просят учесть **служебные данные**: заголовки пакетов;
* встречается расчёт **пропускной способности**: сколько данных
  пройдёт за указанное время.

Порядок действий не меняется: всё в биты, поделить, перевести
обратно.

## Смотрим, как это работает

### Пример 1. Что добавляет каждый уровень

In [ ]:
данные = "Привет"
пакет = данные

уровни = [
    ("прикладной", "HTTP-заголовок", 120),
    ("транспортный", "TCP-заголовок", 20),
    ("сетевой", "IP-заголовок", 20),
    ("канальный", "Ethernet-кадр", 18),
]

размер = len(данные.encode("utf-8"))
print(f"Полезные данные: {размер} байт")

for уровень, что, сколько in уровни:
    размер += сколько
    print(f"  + {что:<16} ({сколько:>3} байт) → {размер:>4} байт после уровня «{уровень}»")

print()
print(f"Накладные расходы: {размер - len(данные.encode('utf-8'))} байт "
      f"на {len(данные.encode('utf-8'))} байт данных")

Вот почему короткие сообщения передавать невыгодно: служебных данных
в разы больше, чем полезных. Поэтому их стараются объединять.

### Пример 2. Время передачи

In [ ]:
def время_передачи(объём_байт, скорость_бит):
    return объём_байт * 8 / скорость_бит


задачи = [
    ("файл 1,5 Мбайт по каналу 512 Кбит/с", 1.5 * 1024 * 1024, 512 * 1024),
    ("фото 4 Мбайт по 100 Мбит/с", 4 * 1024 * 1024, 100 * 1_000_000),
    ("архив 700 Мбайт по 1 Гбит/с", 700 * 1024 * 1024, 1_000_000_000),
]

for описание, объём, скорость in задачи:
    t = время_передачи(объём, скорость)
    print(f"{описание:<38} {t:>8.2f} с")

### Пример 3. Сколько пройдёт за время

In [ ]:
скорость = 256 * 1024        # бит/с
секунд = 60

бит = скорость * секунд
байт = бит / 8

print(f"За {секунд} с на скорости 256 Кбит/с пройдёт:")
print(f"  {бит:,.0f} бит")
print(f"  {байт:,.0f} байт = {байт / 1024:,.1f} Кбайт = {байт / 1024 / 1024:.2f} Мбайт")

### Пример 4. Учёт служебных данных

In [ ]:
полезных_байт = 1_000_000
пакет_данных = 1460
заголовки = 58                 # TCP + IP + Ethernet

пакетов = -(-полезных_байт // пакет_данных)
служебных = пакетов * заголовки
всего = полезных_байт + служебных

print(f"Пакетов: {пакетов}")
print(f"Служебных данных: {служебных:,} байт")
print(f"Всего передано: {всего:,} байт")
print(f"Накладные расходы: {служебных / всего * 100:.2f} %")

### Пример 5. TCP против UDP на пальцах

In [ ]:
потери = 0.02          # два процента пакетов теряется
пакетов = 1000

доставлено_udp = round(пакетов * (1 - потери))
переспросов_tcp = round(пакетов * потери)

print("Отправили 1000 пакетов, теряется 2 %:")
print(f"  UDP: дошло {доставлено_udp}, потери просто пропадают")
print(f"  TCP: дойдёт всё, но потребуется {переспросов_tcp} переспросов")
print()
print("Для видеозвонка потерянный кусок звука уже не нужен — поэтому UDP.")
print("Для файла потерянный кусок критичен — поэтому TCP.")

### Пример 6. Путь запроса

In [ ]:
шаги = [
    "браузер спрашивает у DNS адрес example.com",
    "DNS отвечает: 93.184.216.34",
    "браузер открывает TCP-соединение на порт 443",
    "устанавливается защищённое соединение",
    "браузер шлёт HTTP-запрос GET /",
    "сервер отвечает HTTP-ответом с HTML",
    "браузер разбирает HTML и запрашивает картинки и стили",
]

for номер, шаг in enumerate(шаги, 1):
    print(f"{номер}. {шаг}")

## Пробуем сами

### Задача 1. Протокол с гарантией доставки

Какой транспортный протокол гарантирует доставку и порядок пакетов?

In [ ]:
#@title 🧩 Задача 1. Надёжный протокол { display-mode: "form" }
#@markdown Впиши название заглавными буквами
протокол = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("1", протокол, "5c90162e07ce9d27",
         hint="Его ненадёжная пара — UDP.")

### Задача 2. Служба перевода имён

Как называется служба, которая переводит доменное имя в IP-адрес?

In [ ]:
#@title 🧩 Задача 2. Имена и адреса { display-mode: "form" }
#@markdown Впиши название заглавными буквами
служба = "" #@param {type:"string"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("2", служба, "02f0177364eff0bf",
         hint="Три буквы, работает на порту 53.")

### Задача 3. Время передачи

Функция получает объём в байтах и скорость в битах в секунду.
Возвращает время передачи в секундах, округлённое до двух знаков.

In [ ]:
def время(объём_байт, скорость_бит):
    return ...

In [ ]:
si.check("3", время, [
    ((1_572_864, 524288), 24.0),
    ((1024, 8192), 1.0),
    ((100, 800), 1.0),
    ((0, 1000), 0.0),
])

### Задача 4. Сколько данных пройдёт

Функция получает скорость в битах в секунду и время в секундах.
Возвращает объём в **килобайтах**, округлённый до одного знака.

In [ ]:
def объём_за_время(скорость_бит, секунд):
    return ...

In [ ]:
si.check("4", объём_за_время, [
    ((262144, 60), 1920.0),
    ((8192, 1), 1.0),
    ((8192, 10), 10.0),
    ((1000, 0), 0.0),
])

### Задача 5. Пакеты и заголовки

Функция получает объём полезных данных, размер полезной части
пакета и размер заголовков. Возвращает **общий** объём переданного:
полезные данные плюс заголовки всех пакетов.

Последний пакет считается целым пакетом.

In [ ]:
def всего_передано(полезных, пакет, заголовок):
    return ...

In [ ]:
si.check("5", всего_передано, [
    ((1000, 100, 10), 1100),
    ((1001, 100, 10), 1111),
    ((100, 100, 20), 120),
    ((0, 100, 20), 0),
])

### Задача 6. Уровень протокола

Сколько уровней в упрощённой модели стека TCP/IP, которую мы
разобрали?

In [ ]:
#@title 🧩 Задача 6. Уровни модели { display-mode: "form" }
#@markdown Впиши число
уровней = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("6", уровней, "4b227777d4dd1fc6",
         hint="Прикладной, транспортный, сетевой, канальный.")

## Домашнее задание

### Домашнее задание 1. Путь запроса своими словами

Опишите по шагам, что происходит от момента, когда вы вводите
адрес сайта, до появления страницы. Для каждого шага укажите,
какой уровень модели и какой протокол работает.

Отдельно объясните, что сломается, если не работает только DNS.
Покажите работу учителю.

### Домашнее задание 2. Скорость по времени

Функция получает объём в байтах и время в секундах. Возвращает
скорость канала в килобитах в секунду, округлённую до одного знака.

In [ ]:
def скорость(объём_байт, секунд):
    return ...

In [ ]:
si.check("дз2", скорость, [
    ((1_572_864, 24), 512.0),
    ((1024, 1), 8.0),
    ((131072, 2), 512.0),
])

### Домашнее задание 3. Сколько бит в байте

На какой множитель нужно умножить объём в байтах, чтобы получить
биты?

In [ ]:
#@title 🏠 Домашнее 3. Множитель { display-mode: "form" }
#@markdown Впиши число
множитель = 0 #@param {type:"integer"}

if "si" not in globals(): raise SystemExit("⚠️ Сначала запусти ячейку «🚀 Регистрация» в начале урока (Shift+Enter) и дождись, пока она закончит работу.")
si.ответ("дз3", множитель, "2c624232cdd22177",
         hint="Самая частая потеря балла в задачах на скорость.")

---

### Любопытно

Уровневая модель родилась из спора. В 1970-х конкурировали две
системы: изящная семиуровневая OSI, разработанная международным
комитетом, и грубоватый четырёхуровневый TCP/IP, выросший
из практики ARPANET.

Комитет писал стандарты годами. Авторы TCP/IP раздавали работающий
код бесплатно вместе с операционной системой Unix. К моменту, когда
OSI довели до конца, интернет уже работал на TCP/IP — и переписывать
его никто не стал.

Модель OSI осталась в учебниках как способ объяснять уровни,
а работает везде TCP/IP. Это классическая история про то,
что побеждает не самое красивое, а то, что успели запустить.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 18](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-18.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 20 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-20.ipynb)